# **SNC - Signaling**

In [26]:
import pandas as pd
import json, re

In [27]:
path = "../../output/snc/signalling.xlsx"

df_signaling = pd.read_excel(path, sheet_name="signalling", keep_default_na=False)
df_signaling_wayside = pd.read_excel(path, sheet_name="signalling_wayside", keep_default_na=False)
df_ctc = pd.read_excel(path, sheet_name="ctc", keep_default_na=False)

df_signaling = df_signaling.add_prefix("signaling.")
df_signaling.rename(columns={"signaling.filename": "filename"}, inplace=True)

df_signaling_wayside = df_signaling_wayside.add_prefix("wayside.")
df_signaling_wayside.rename(columns={"wayside.filename": "filename"}, inplace=True)

df_ctc = df_ctc.add_prefix("ctc.")
df_ctc.rename(columns={"ctc.filename": "filename"}, inplace=True)

In [28]:
def extract_workorder_id(filename):
    if not filename or pd.isna(filename):
        return None

    if re.search(r"_NA(_|\.)", filename):
        return None

    match = re.search(r"_(\d+)\.pdf$", filename)
    if match:
        return match.group(1)

    return None

def df_to_filename_json(df):
    rows = {}
    for _, row in df.iterrows():
        filename = row["filename"]
        data = row.drop("filename").to_dict()
        rows[filename] = data
    return rows

signalling_map = df_to_filename_json(df_signaling)
wayside_map = df_to_filename_json(df_signaling_wayside)
ctc_map = df_to_filename_json(df_ctc)

all_filenames = set(signalling_map) | set(wayside_map) | set(ctc_map)

rows = []

for filename in all_filenames:
    rows.append({
        "workorder_id": extract_workorder_id(filename),
        "filename": filename,
        "signalling": json.dumps(signalling_map.get(filename, {}), ensure_ascii=False),
        "signalling_wayside": json.dumps(wayside_map.get(filename, {}), ensure_ascii=False),
        "ctc": json.dumps(ctc_map.get(filename, {}), ensure_ascii=False),
    })

df_out = pd.DataFrame(rows)

output_file = "../../output/snc/signaling_combined.xlsx"
df_out.to_excel(output_file, index=False)
print(f"Saved as: {output_file}")



Saved as: ../../output/snc/signaling_combined.xlsx
